In [ ]:
# ── Gerekli Kütüphaneler ──────────────────────────────────────────────────────
# Unsloth'u ÖNCE kur, sonra diğerlerini
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes -q
!pip install datasets -q

In [ ]:
import torch
import json
import math
import random
import matplotlib.pyplot as plt
from datasets import load_dataset
from unsloth import FastLanguageModel   # ← Artık aktif kullanıyoruz
from trl import SFTConfig, SFTTrainer

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
#veri seti yükleme

full_dataset = load_dataset("json", data_files="analiz_sbertsiz.jsonl", split="train")
print(f"Toplam örnek sayısı: {len(full_dataset)}")

# Önce %10 test ayır
split_1 = full_dataset.train_test_split(test_size=0.10, seed=42)
train_val    = split_1["train"]
test_dataset = split_1["test"]

# Kalanın %11.1'i → toplam verinin %10'u validation olur
split_2       = train_val.train_test_split(test_size=0.111, seed=42)
train_dataset = split_2["train"]
val_dataset   = split_2["test"]

print(f"\nVeri dağılımı:")
print(f"  Train      : {len(train_dataset)} örnek ({len(train_dataset)/len(full_dataset)*100:.1f}%)")
print(f"  Validation : {len(val_dataset)} örnek ({len(val_dataset)/len(full_dataset)*100:.1f}%)")
print(f"  Test       : {len(test_dataset)} örnek ({len(test_dataset)/len(full_dataset)*100:.1f}%)")

QWEN 2.5 İÇİN

In [ ]:
# ── Ortak Parametreler ────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 2048   # Her iki model için aynı
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
MAX_STEPS      = 500

# LoRA hedef modüller (Qwen ve Llama için ortak isimler)
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

def get_training_args(output_dir):
    """Her iki model için aynı eğitim ayarları — adil karşılaştırma."""
    return SFTConfig(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        optim="adamw_8bit",             # Unsloth ile uyumlu optimizer
        learning_rate=2e-4,
        max_steps=MAX_STEPS,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        # ── Validation Ayarları ──────────────────
        eval_strategy="steps", # Changed evaluation_strategy to eval_strategy
        eval_steps=50,
        save_steps=100,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        # ─────────────────────────────────────────
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        push_to_hub=False,
        report_to="none"
    )

def plot_training_history(log_history, model_name):
    """Train ve Validation loss grafiğini çizer."""
    train_steps, train_losses = [], []
    eval_steps,  eval_losses  = [], []

    for entry in log_history:
        if "loss" in entry and "eval_loss" not in entry:
            train_steps.append(entry["step"])
            train_losses.append(entry["loss"])
        if "eval_loss" in entry:
            eval_steps.append(entry["step"])
            eval_losses.append(entry["eval_loss"])

    plt.figure(figsize=(10, 5))
    plt.plot(train_steps, train_losses, label="Train Loss",      alpha=0.7)
    plt.plot(eval_steps,  eval_losses,  label="Validation Loss", marker="o")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title(f"{model_name} — Eğitim & Validation Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{model_name.replace('/', '_')}_loss.png", dpi=150)
    plt.show()
    print(f"Son Train Loss : {train_losses[-1]:.4f}")
    print(f"Son Val Loss   : {eval_losses[-1]:.4f}")

print("Yardımcı fonksiyonlar hazır ✓")

Modeli Yükleme

In [ ]:
# ── QWEN 2.5 Yükleme ────────────────────────────────────────────────────────────

QWEN25_MODEL_NAME="Qwen/Qwen2.5-3B-Instruct"

qwen25_model, qwen25_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = QWEN25_MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = True,          # 4-bit quantization (BnbConfig'e gerek yok)
    dtype          = None,          # Otomatik tespit (bf16 destekliyorsa bf16)
)

qwen25_model = FastLanguageModel.get_peft_model(
    qwen25_model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    target_modules = LORA_TARGET_MODULES,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth'a özgü, %30 daha az VRAM
    random_state   = 42,
)

qwen25_model.print_trainable_parameters()

Veri Setini EOS ile Formatlama

In [ ]:
# ── Alpaca Format Dönüşümü ────────────────────────────────────────────────────
def formatting_func(example):
    text = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n{example['output']}{qwen25_tokenizer.eos_token}"
    )
    return {"text": text}

train_dataset = train_dataset.map(formatting_func, remove_columns=train_dataset.column_names)
val_dataset   = val_dataset.map(formatting_func,   remove_columns=val_dataset.column_names)
test_dataset  = test_dataset.map(formatting_func,  remove_columns=test_dataset.column_names)

print("Format örneği:")
print(train_dataset[0]["text"][:400])

Eğitim

In [ ]:
# ── Qwen2.5 Eğitim ─────────────────────────────────────────────────────────────
qwen25_trainer = SFTTrainer(
    model         = qwen25_model,
    tokenizer     = qwen25_tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,    # ← Validation seti
    args          = get_training_args("./qwen25_finetuned_sbertsiz"),
)

# VRAM durumunu göster
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name} | VRAM: {gpu_stats.total_memory / 1e9:.1f} GB")
print(f"Eğitim öncesi ayrılan VRAM: {start_gpu_memory} GB")
print("\nQwen2.5 eğitimi başlıyor...")

qwen25_result = qwen25_trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\nQwen2.5 eğitimi tamamlandı ✓")
print(f"Süre         : {qwen25_result.metrics['train_runtime']:.1f} sn")
print(f"Peak VRAM    : {used_memory} GB")

Grafik

In [ ]:
# ── Qwen2.5 Sonuçları ───────────────────────────────────────────────────────────
plot_training_history(qwen25_trainer.state.log_history, "Qwen2.5-3B Sbertsiz İnce Ayar")

Ağırlıkları Kaydetme Drive

In [ ]:
# ── Qwen2.5 Kaydetme ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

QWEN_SAVE_DIR = "/content/drive/MyDrive/QWEN25_finetuned_eos_sbertsiz"
Path(QWEN_SAVE_DIR).mkdir(parents=True, exist_ok=True)

qwen25_model.save_pretrained(QWEN_SAVE_DIR)   # LoRA ağırlıkları
qwen25_tokenizer.save_pretrained(QWEN_SAVE_DIR)
print(f"Qwen2.5 kaydedildi: {QWEN_SAVE_DIR}")

# Validation metriklerini kaydet (karşılaştırma için)
qwen_metrics = {e["step"]: e["eval_loss"] for e in qwen25_trainer.state.log_history if "eval_loss" in e}
with open("/content/drive/MyDrive/qwen25_val_metrics_sbertsiz.json", "w") as f:
    json.dump(qwen_metrics, f, indent=2)
print("Validation metrikleri kaydedildi ✓")

Ağırlıkları Kaydetme İndirme

In [ ]:
# ── Qwen2.5 Lokal İndirme ───────────────────────────────────────────────────────
import shutil
import json
from google.colab import files
from pathlib import Path

# 1. Modeli geçici klasöre kaydet
QWEN_SAVE_DIR = "/content/QWEN25_finetuned_eos_sbertsiz"
Path(QWEN_SAVE_DIR).mkdir(parents=True, exist_ok=True)

qwen25_model.save_pretrained(QWEN_SAVE_DIR)
qwen25_tokenizer.save_pretrained(QWEN_SAVE_DIR)
print(f"Qwen2.5 kaydedildi: {QWEN_SAVE_DIR}")

# 2. Validation metriklerini kaydet
qwen_metrics = {e["step"]: e["eval_loss"] for e in qwen25_trainer.state.log_history if "eval_loss" in e}
metrics_path = "/content/qwen25_val_metrics_sbertsiz.json"
with open(metrics_path, "w") as f:
    json.dump(qwen_metrics, f, indent=2)
print("Validation metrikleri kaydedildi ✓")

# 3. Klasörü zip'le
zip_path = "/content/QWEN25_finetuned_eos_sbertsiz"
shutil.make_archive(zip_path, "zip", QWEN_SAVE_DIR)
print(f"Zip oluşturuldu: {zip_path}.zip")

# 4. İndir
files.download(f"{zip_path}.zip")
files.download(metrics_path)
print("İndirme başladı ✓")